# Verificación de ejemplos JSON

Este notebook carga los ejemplos JSON generados en `json_examples/` y valida que sean compatibles con los módulos de inferencia de `m07` y `m09`.

- Comprueba que existan 60 archivos
- Verifica `ejercicio_id` y el contenido de `secuencia`
- Normaliza los frames con `coerce_frames`
- Extrae la ventana biomecánica con `extract_biomechanical_window`
- Muestra estadísticas de cada ejercicio

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np

ROOT_DIR = Path(".").resolve()
EXAMPLES_DIR = ROOT_DIR / "json_examples"
MODULE_PATHS = {
    7: ROOT_DIR / "standing_shoulder_internal_external_rotation" / "Standing_Shoulder_Internal_External_Rotation.py",
    9: ROOT_DIR / "standing_shoulder_abduction" / "Standing_Shoulder_Abduction.py",
}


def load_module(path: Path, name: str) -> Any:
    import importlib.util

    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"No se pudo cargar el módulo {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def parse_exercise_id(payload: dict[str, Any]) -> int:
    raw = payload.get("ejercicio_id")
    if raw is None:
        raw = payload.get("movement_id")
    if isinstance(raw, int):
        return raw
    if isinstance(raw, str):
        val = raw.strip().lower().replace("m", "")
        if val.isdigit():
            return int(val)
    raise ValueError("No se pudo determinar el ejercicio.")

## Funciones de validación

Definimos las funciones que cargan los archivos JSON, validan los datos y ejecutan las comprobaciones básicas.

In [2]:
def validate_payload_file(path: Path, module: Any) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if not isinstance(payload, dict):
        raise AssertionError(f"Payload no es un dict en {path}")

    exercise_id = parse_exercise_id(payload)
    assert exercise_id in MODULE_PATHS, f"Ejercicio no soportado en {path}: {exercise_id}"
    assert any(key in payload for key in ("secuencia", "frames", "ventana", "window")), f"Falta secuencia en {path}"

    frames = module.coerce_frames(payload)
    assert isinstance(frames, list), f"Frames normalizados no son lista en {path}"
    assert frames, f"No se encontraron frames normalizados en {path}"
    assert all("puntos_clave" in frame for frame in frames), f"Falta puntos_clave en algún frame de {path}"

    window_matrix, summary = module.extract_biomechanical_window(frames)
    assert isinstance(window_matrix, np.ndarray), f"Window matrix no es np.ndarray en {path}"
    assert window_matrix.shape[0] == module.WINDOW_SIZE, f"Window size incorrecto en {path}: {window_matrix.shape[0]}"
    assert window_matrix.shape[1] == len(module.FEATURE_NAMES), f"Número de features incorrecto en {path}: {window_matrix.shape[1]}"
    assert isinstance(summary, dict), f"Summary no es dict en {path}"

    return {
        "path": path,
        "exercise_id": exercise_id,
        "n_frames": len(frames),
        "window_shape": window_matrix.shape,
        "summary_keys": list(summary.keys()),
    }


def collect_example_files() -> list[Path]:
    return sorted(EXAMPLES_DIR.rglob("*.json"))


## Validación y resumen

Ejecutamos la validación de todos los ejemplos, agrupados por ejercicio, y mostramos estadísticas de cada conjunto.

In [3]:
example_files = collect_example_files()
print(f"Archivos JSON encontrados: {len(example_files)}")
assert len(example_files) >= 60, "No se encontraron al menos 60 ejemplos JSON."

results: list[dict[str, Any]] = []
modules_cache: dict[int, Any] = {}

for path in example_files:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    exercise_id = parse_exercise_id(payload)
    if exercise_id not in modules_cache:
        modules_cache[exercise_id] = load_module(MODULE_PATHS[exercise_id], f"validation_m{exercise_id}")
    module = modules_cache[exercise_id]
    result = validate_payload_file(path, module)
    results.append(result)

summary = {
    7: [r for r in results if r["exercise_id"] == 7],
    9: [r for r in results if r["exercise_id"] == 9],
}

for eid, items in summary.items():
    print(f"\nEjercicio m{eid:02d}: {len(items)} ejemplos")
    if items:
        lengths = [item["n_frames"] for item in items]
        print(f" - n_frames: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.1f}")
        shapes = sorted({item["window_shape"] for item in items})
        print(f" - window shapes: {shapes}")

print("\nValidación completada sin errores.")

Archivos JSON encontrados: 60

Ejercicio m07: 30 ejemplos
 - n_frames: min=37, max=59, avg=47.2
 - window shapes: [(48, 10)]

Ejercicio m09: 30 ejemplos
 - n_frames: min=36, max=59, avg=47.6
 - window shapes: [(48, 10)]

Validación completada sin errores.
